In [ ]:
import torch, random
import numpy as np
from pathlib import Path
from PIL import Image, ImageOps
from ldm.models.autoencoder import AutoencoderKL

In [ ]:
IMAGENET_DIR=Path("/raid/datasets/imagenet")
# if multiple GPUs are present, diversify their usage
device_id = random.randint(0, torch.cuda.device_count() - 1)
device = f"cuda:{device_id}"

In [ ]:
sd = torch.load("logs/2025-10-18T18-29-47_autoencoder_kl_32x32x4/checkpoints/epoch=000149.ckpt", map_location="cpu")['state_dict']

vae_params = {
    "ddconfig": {'double_z': True, 'z_channels': 4, 'resolution': 256, 'in_channels': 3, 'out_ch': 3, 'ch': 128, 'ch_mult': [1, 2, 4, 4],
                  'num_res_blocks': 2, 'attn_resolutions': [], 'dropout': 0.0},
    "lossconfig": {'target': 'ldm.modules.losses.LPIPSWithDiscriminator', 'params': {'disc_start': 50001, 'kl_weight': 1e-06, 'disc_weight': 0.5}},
    "embed_dim": 4,
}
model = AutoencoderKL(**vae_params)
model.load_state_dict(sd)
_ = model.to(device)

In [ ]:
synset = "n03733281"
valdir = IMAGENET_DIR / "val" / synset
val_images = [x for x in valdir.iterdir() if x.is_file()]
len(val_images)

In [ ]:
im = Image.open(val_images[14])
im = ImageOps.fit(im, (256, 256), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
display(im)

arr = np.array(im)
x = torch.from_numpy(arr).to(device)
x = (x/127.5 - 1).unsqueeze(0).permute(0,3,1,2)

In [ ]:
y, _ = model(x)
y = ((y + 1) * 127.5).to(torch.uint8)[0].permute(1,2,0)
out = Image.fromarray(y.detach().cpu().numpy())
display(out)